Install the Python libraries we need. We also install `curl`, which the agent uses as a tool.

In [ ]:
!apt update && apt install -y curl
%pip install fastapi==0.135.3
%pip install kaggle-benchmarks==0.3.0
%pip install python-dotenv==1.2.2
%pip install requests==2.33.0
%pip install uvicorn==0.44.0

We provide this benchmark source code as a Kaggle dataset. Add the dataset folder to `sys.path` so Python can import from it, then import everything required.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/input/datasets/andgein/russian-doll-source-code")

In [ ]:
import shutil
import kaggle_benchmarks as kbench
from kaggle_benchmarks import envs
from kaggle_benchmarks.kaggle import models

from benchmark.envs import LocalEnvironment
from benchmark.infrastructure import VirtualFileSystem, get_log_filename
from benchmark.telemetry import enable_logging, set_log_file
from benchmark.tasks import run_adaptive_learning

Set up the environment and run the task.

In [ ]:
# By default, if kbench.llm points to Google Gemini/Gemma, it's created as OpenAI-compatible model (see load_default_model()).
# Let's recreate it as a "genai" model to use a native client.
if kbench.llm.name.startswith("google/"):
    kbench.llm = models.load_model(model_name=kbench.llm.name, api="genai")

# Configure environment
enable_logging()
set_log_file("/kaggle/working/" + get_log_filename("kaggle", kbench.llm.name))
envs.current = LocalEnvironment()
VirtualFileSystem.override_root(envs.current.directory)

# Run task
try:
    run_adaptive_learning(kbench.llm)
finally:
    shutil.copytree(envs.current.directory, "/kaggle/working/working-dir", dirs_exist_ok=True)